# Data Preprocessing

This notebook prepares the ScienceQA-IMG dataset for text-only and multimodal model development.

The preprocessing uses BERT for text and ViT for image feature extraction.

In [5]:

from datasets import load_from_disk

clean_dataset = load_from_disk(
    "../data/processed/scienceqa_cleaned"
)

for split in clean_dataset:
    print(split, len(clean_dataset[split]))


train 3530
validation 1343
test 2017


In [6]:

from transformers import AutoTokenizer, AutoImageProcessor

BERT_MODEL = "bert-base-uncased"
VIT_MODEL = "google/vit-base-patch16-224"

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
image_processor = AutoImageProcessor.from_pretrained(VIT_MODEL)

print("BERT tokenizer loaded successfully")
print("ViT image processor loaded successfully")


W1110 21:58:37.271000 6794 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

BERT tokenizer loaded successfully
ViT image processor loaded successfully


In [7]:

sample = clean_dataset["train"][0]

# Text preprocessing
text = sample["question"] + " [SEP] " + sample["choices"][0]

tokens = tokenizer(
    text,
    truncation=True,
    max_length=128,
    padding="max_length",
    return_tensors="pt"
)

# Image preprocessing
image_inputs = image_processor(
    images=sample["image"].convert("RGB"),
    return_tensors="pt"
)

print("Text shape:", tokens["input_ids"].shape)
print("Image shape:", image_inputs["pixel_values"].shape)


Text shape: torch.Size([1, 128])
Image shape: torch.Size([1, 3, 224, 224])


## question-option pairs

In [8]:

sample = clean_dataset["train"][0]

for i, choice in enumerate(sample["choices"]):
    text = sample["question"] + " [SEP] " + choice

    encoded = tokenizer(
        text,
        truncation=True,
        max_length=128,
        padding="max_length",
        return_tensors="pt"
    )

    print(f"Option {i + 1}: {choice}")
    print("Token shape:", encoded["input_ids"].shape)

print("\nCorrect answer index:", sample["answer"])
print("Number of choices:", len(sample["choices"]))


Option 1: West Virginia
Token shape: torch.Size([1, 128])
Option 2: Louisiana
Token shape: torch.Size([1, 128])
Option 3: Arizona
Token shape: torch.Size([1, 128])
Option 4: Oklahoma
Token shape: torch.Size([1, 128])

Correct answer index: 0
Number of choices: 4


## Complete Dataset Preprocessing

Each question is represented by its image and up to five question-option text pairs. Optional hints are included when available. Padding and option masks handle variable numbers of answer choices.

In [9]:

import numpy as np

MAX_LENGTH = 128
MAX_CHOICES = 5

def preprocess_sample(sample):

    # Process image once per question
    image = image_processor(
        images=sample["image"].convert("RGB"),
        return_tensors="np"
    )["pixel_values"][0]

    input_ids = []
    attention_masks = []
    option_mask = []

    # Process each answer choice
    for choice in sample["choices"]:

        text = (
            f"Question: {sample['question']} "
            f"Hint: {sample['hint'] or 'None'} "
            f"Answer: {choice}"
        )

        encoded = tokenizer(
            text,
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length"
        )

        input_ids.append(encoded["input_ids"])
        attention_masks.append(encoded["attention_mask"])
        option_mask.append(1)

    # Pad to five choices
    while len(input_ids) < MAX_CHOICES:
        input_ids.append([0] * MAX_LENGTH)
        attention_masks.append([0] * MAX_LENGTH)
        option_mask.append(0)

    return {
        "pixel_values": image.tolist(),
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        "option_mask": option_mask,
        "labels": sample["answer"]
    }


In [10]:

sample = clean_dataset["train"][0]

processed = preprocess_sample(sample)

for key, value in processed.items():
    print(key, np.array(value).shape)


pixel_values (3, 224, 224)
input_ids (5, 128)
attention_mask (5, 128)
option_mask (5,)
labels ()


## Process the complete dataset

In [11]:

from tqdm.auto import tqdm
from datasets import DatasetDict

processed_dataset = {}

for split in ["train", "validation", "test"]:

    print(f"\nProcessing {split}...")

    processed_dataset[split] = clean_dataset[split].map(
        preprocess_sample,
        remove_columns=clean_dataset[split].column_names,
        desc=f"Preprocessing {split}"
    )

processed_dataset = DatasetDict(processed_dataset)

print("\nPreprocessing completed!")

for split in processed_dataset:
    print(split, len(processed_dataset[split]))



Processing train...


Preprocessing train:   0%|          | 0/3530 [00:00<?, ? examples/s]


Processing validation...


Preprocessing validation:   0%|          | 0/1343 [00:00<?, ? examples/s]


Processing test...


Preprocessing test:   0%|          | 0/2017 [00:00<?, ? examples/s]


Preprocessing completed!
train 3530
validation 1343
test 2017


In [12]:
processed_dataset.save_to_disk(
    "../data/processed/scienceqa_preprocessed"
)

Saving the dataset (0/9 shards):   0%|          | 0/3530 [00:00<?, ? examples/s]

Saving the dataset (0/4 shards):   0%|          | 0/1343 [00:00<?, ? examples/s]

Saving the dataset (0/5 shards):   0%|          | 0/2017 [00:00<?, ? examples/s]

Verification of the saved preprocess

In [13]:

from datasets import load_from_disk

processed_dataset = load_from_disk(
    "../data/processed/scienceqa_preprocessed"
)

for split in processed_dataset:
    print(split, len(processed_dataset[split]))

print("\nSample feature shapes:")

sample = processed_dataset["train"][0]

for key, value in sample.items():
    print(key, np.array(value).shape)


train 3530
validation 1343
test 2017

Sample feature shapes:
pixel_values (3, 224, 224)
input_ids (5, 128)
attention_mask (5, 128)
option_mask (5,)
labels ()
